In [1]:
from redis import Redis

redis = Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

In [36]:
redis.lpush("summary_queue", "session:15")

1

In [29]:
redis.lpush("summary_queue", "session:22")
redis.lpush("summary_queue", "session:35")

1

In [25]:
redis.lpush("summary_queue", "session:90")

1

In [30]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy import desc
from uuid import UUID

# مدل‌ها را ایمپورت کن
from app.models.database import Message
from app.models.database  import Session

# دیتابیس
from app.core.database.database import APP_DATABASE_URL

engine = create_engine(APP_DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

db = SessionLocal()

# آیدی سشن
session_id = UUID("d4b24554-a33f-4f35-901c-78a6313d0492")

# گرفتن 10 پیام آخر
messages = (
    db.query(Message)
    .filter(Message.session_id == session_id)
    .order_by(desc(Message.created_at))
    .limit(10)
    .all()
)

# اگر بخواهی از قدیمی به جدید نمایش داده شوند
messages = list(reversed(messages))

for msg in messages:
    print("=" * 80)
    # print(f"ID: {msg.id}")
    print(f"Role: {msg.role}")
    # print(f"Created At: {msg.created_at}")
    print(msg.content)

Role: user
سلام حالت چطوره ؟ تو چی هستی
Role: agent
من یک دستیار هوشمند هستم که برای پاسخ به سوالات شما در زمینه سیستم‌های RAG طراحی شده‌ام. اگر سوالی درباره این موضوع دارید، خوشحال می‌شوم کمک کنم.
Role: user
بیشتر راهنماییم کن ، اسم منم حامد هستش 
Role: agent
حامد جان، خوشوقتم. درباره سیستم‌های RAG چطور می‌خواهی راهنمایی بشی؟ می‌تونم درباره مفاهیم پایه، معماری، کاربردها یا نحوه پیاده‌سازی‌شون توضیح بدم. لطفاً بگو دقیقاً دنبال چه اطلاعاتی هستی تا بهتر کمکت کنم.
Role: user
میخوام بدونم این سیستم ها چه کاربردهایی دارند ؟
Role: agent
حامد جان، سیستم‌های RAG کاربردهای متنوع و مهمی دارند که به دلیل توانایی‌شان در ترکیب بازیابی اطلاعات مرتبط با تولید پاسخ دقیق، در حوزه‌های مختلف به کار می‌روند. بر اساس بخش [1.1] مقدمه و پیشینه در مستندات، برخی از کاربردهای اصلی این سیستم‌ها عبارتند از:

- پرسش و پاسخ سازمانی: کمک به پاسخگویی دقیق به سوالات کاربران با استفاده از داده‌های داخلی سازمان.
- چت بات‌های پشتیبانی مشتری: ارائه پاسخ‌های سریع و مرتبط به مشتریان با دسترسی به پایگاه دانش شرکت.
- تحلیل اس

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

from app.core.config import settings

BASE_URL = "https://api.gapgpt.app/v1"
API_KEY = settings.OPENAI_API_KEY

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    base_url=BASE_URL,
    api_key=API_KEY,
)

# تبدیل پیام‌ها به متن
conversation = "\n\n".join(
    f"{msg.role.upper()}:\n{msg.content}"
    for msg in messages
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
تو وظیفه داری مکالمه زیر را تحلیل کنی.

فقط یک خلاصه کوتاه و مفید از موضوعات مطرح‌شده ارائه بده.

قوانین:
- حداکثر 150 کلمه.
- جزئیات غیرضروری را حذف کن.
- اگر چند موضوع وجود دارد، همه را پوشش بده.
- متن خلاصه باید به فارسی باشد.
- فقط خلاصه را برگردان و هیچ توضیح اضافه‌ای ننویس.
            """,
        ),
        ("human", "{conversation}"),
    ]
)

chain = prompt | llm | StrOutputParser()

summary = chain.invoke({"conversation": conversation})

print(summary)

مکالمه درباره سیستم‌های RAG و کاربردهای آن بود. دستیار هوشمند به حامد توضیح داد که این سیستم‌ها در حوزه‌های مختلفی مانند پرسش و پاسخ سازمانی، چت‌بات‌های پشتیبانی، تحلیل اسناد حقوقی، سیستم‌های پزشکی و هوش تجاری کاربرد دارند. سپس به طور دقیق‌تر درباره کاربردهای RAG در هوش تجاری توضیح داده شد، از جمله تحلیل خودکار گزارش‌ها، پاسخ به پرسش‌های مدیریتی، تولید داشبوردهای هوشمند و استخراج بینش‌های ارزشمند برای بهبود تصمیم‌گیری.


In [22]:
import markdown
from IPython.display import display, HTML

def print_md(content):
    content = markdown.markdown(content)
    content = "<div dir=rtl>{}</div>".format(content)
    display(HTML(content))

print_md(summary)

In [2]:
import markdown
from IPython.display import display, HTML

def print_md(content):
    content = markdown.markdown(content)
    content = "<div dir=rtl>{}</div>".format(content)
    display(HTML(content))

In [10]:
s1=" سیستم‌های RAG می‌توانند در پروژه شما به این شکل کمک کنند:\n\n- تحلیل خودکار گزارش‌ها و کاهش زمان بررسی داده‌ها  \n- پاسخ سریع به پرسش‌های مدیریتی با استفاده از داده‌های داخلی بانک  \n- تولید داشبوردهای هوشمند با توضیحات متنی که فهم داده‌ها را آسان‌تر می‌کند  \n- استخراج بینش‌های مهم از داده‌های ساخت‌یافته و غیرساخت‌یافته بانک  \n\nاین موارد باعث افزایش دقت و سرعت تصمیم‌گیری در پروژه‌های BI شما می‌شوند. (بخش کاربرد RAG در هوش تجاری)"
print_md(s1)